# Python 函数名作为参数 & 返回函数名（高阶函数 / 闭包）
在 Python 中，函数是一等对象（First-Class Object） —— 这意味着函数可以像普通变量一样：赋值给变量、作为参数传递给其他函数、作为返回值返回。这是实现高阶函数、闭包、装饰器的核心基础，下面分两部分详细讲解并结合示例说明。
## 一、函数名作为参数（高阶函数）
1. 核心概念
接收另一个函数作为参数的函数，称为「高阶函数」。函数名本质是指向函数对象的「引用」，传递函数名就是传递这个引用，在函数内部可以调用传入的函数。
2. 示例1:自定义高阶函数（直观）

In [1]:
# 定义一个普通函数：计算平方
def square(x):
    return x * x

# 定义一个普通函数：计算立方
def cube(x):
    return x * x * x

# 定义高阶函数：接收函数和数字，应用该函数到数字上
def apply_func(func, num):
    """
    :param func: 传入的函数（函数名）
    :param num: 要计算的数字
    :return: 执行 func(num) 的结果
    """
    print(f"调用函数：{func.__name__}，参数：{num}")  # __name__ 获取函数名
    return func(num)

# 调用高阶函数：传入 square 函数名
result1 = apply_func(square, 5)
print("平方结果：", result1)  # 输出：平方结果：25

# 调用高阶函数：传入 cube 函数名
result2 = apply_func(cube, 5)
print("立方结果：", result2)  # 输出：立方结果：125

调用函数：square，参数：5
平方结果： 25
调用函数：cube，参数：5
立方结果： 125


3. 示例2:内置高阶函数（常用）
Python 内置的 map()、filter()、sorted() 都是典型的高阶函数，核心是接收函数作为参数：

In [2]:
# 示例：map(func, 可迭代对象) —— 对每个元素应用 func
nums = [1, 2, 3, 4]
# 传入 square 函数，计算每个元素的平方
square_nums = list(map(square, nums))
print("map 结果：", square_nums)  # 输出：map 结果：[1, 4, 9, 16]

# 示例：filter(func, 可迭代对象) —— 按 func 条件过滤元素
def is_even(x):
    return x % 2 == 0

even_nums = list(filter(is_even, nums))
print("filter 结果：", even_nums)  # 输出：filter 结果：[2, 4]

map 结果： [1, 4, 9, 16]
filter 结果： [2, 4]


4. 关键解释
- 函数名本身是一个【变量】指向函数对象
- 传递函数名(square)而非(square())：square()是执行函数并返回结果，square是传递函数对象本身。
- 高阶函数的价值：将「操作逻辑」与「数据」分离，提高代码复用性。

## 二、返回函数名（返回函数/闭包）
1. 核心概念
函数执行后，返回另一个函数（函数名 / 函数对象） 而非具体值，返回的函数可以在外部调用，且能访问外层函数的变量（这一特性称为「闭包」）。
2. 示例1:简单返回函数

In [3]:
# 定义外层函数：生成加法器
def create_adder(add_num):
    """
    :param add_num: 要加的固定数值
    :return: 一个加法函数（接收一个参数 x，返回 x + add_num）
    """
    # 定义内层函数（要返回的函数）
    def adder(x):
        return x + add_num

    # 返回内层函数名（不执行，仅返回函数对象）
    return adder

# 调用外层函数：生成「加5」的函数
add5 = create_adder(5)
print("add5 的类型：", type(add5))  # 输出：add5 的类型：<class 'function'>
print("add5(10) =", add5(10))      # 输出：add5(10) = 15

# 调用外层函数：生成「加10」的函数
add10 = create_adder(10)
print("add10(10) =", add10(10))    # 输出：add10(10) = 20

add5 的类型： <class 'function'>
add5(10) = 15
add10(10) = 20


3. 示例2:闭包（带状态的返回函数）
返回的函数会「记住」外层函数的变量（即使外层函数已执行完毕），这就是闭包的核心价值 ——保存函数的执行状态：

In [4]:
# 定义计数器生成函数
def create_counter():
    # 外层变量（被内层函数引用，形成闭包）
    count = 0

    # 内层函数：修改并返回 count
    def counter():
        nonlocal count  # 声明使用外层函数的变量（否则 count 会被视为局部变量）
        count += 1
        return count

    return counter

# 创建计数器1（独立状态）
counter1 = create_counter()
print(counter1())  # 输出：1
print(counter1())  # 输出：2

# 创建计数器2（独立状态，与 counter1 互不影响）
counter2 = create_counter()
print(counter2())  # 输出：1
print(counter1())  # 输出：3（counter1 的状态继续累加）

1
2
1
3


4. 关键解释
- 返回 adder/counter 而非 adder()/counter()：返回函数对象（未执行），外部调用时才执行；
- 闭包的条件：
    1. 内层函数引用外层函数的变量；
    2. 外层函数返回内层函数；
- nonlocal 关键字：用于修改外层函数的变量（若仅读取，可省略；若修改，必须声明）；
- 闭包的价值：无需全局变量，即可保存函数的状态（如示例中的计数器）。

## 综合示例

In [5]:
# 定义基础操作函数
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

# 定义高阶函数：接收操作函数，返回「带日志的操作函数」
def with_log(func):
    """
    :param func: 传入的操作函数（add/multiply）
    :return: 带日志的新函数
    """
    # 内层函数：接收操作函数的参数
    def wrapper(a, b):
        print(f"执行 {func.__name__} 操作，参数：{a}, {b}")
        result = func(a, b)  # 调用传入的函数
        print(f"{func.__name__} 结果：{result}")
        return result

    return wrapper

# 1. 传入 add 函数，返回带日志的 add 函数
log_add = with_log(add)
log_add(3, 5)  # 执行 add 操作，参数：3, 5 → add 结果：8

# 2. 传入 multiply 函数，返回带日志的 multiply 函数
log_multiply = with_log(multiply)
log_multiply(3, 5)  # 执行 multiply 操作，参数：3, 5 → multiply 结果：15

执行 add 操作，参数：3, 5
add 结果：8
执行 multiply 操作，参数：3, 5
multiply 结果：15


15